# Carregar Token

In [1]:
%run ./config_api_acto

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 3, Finished, Available, Finished)

# Imports e Configuração

In [2]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone
import numpy as np

# Ambiente: https://gestaoaprovasantos.acto.net.br/#/home
TOKEN = TOKEN_SANTOS_OBRAS

HOJE = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

PARAM_LOGIN = "3997"  
ORIGIN_URL = "https://gestaoaprovasantos.acto.net.br"
APP_ID = "86bf9fc6-78ad-4a65-89e8-8c91f8eac43d"
BASE_API_URL = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net"

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": f"Bearer {TOKEN}",
    "App_Id": APP_ID,
    "ApplicationId": APP_ID,
    "Origin": ORIGIN_URL,
    "Referer": f"{ORIGIN_URL}/",
    "PARAM_LOGIN": PARAM_LOGIN,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Content-Type": "application/json",
}

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 4, Finished, Available, Finished)

<!-- Célula 2: Funções de API
listar_catalogos() - Lista catálogos disponíveis
obter_dados_etapa_atual(TOKEN, codigo_catalogo) - Obtém dados de etapas
fetch_tabela(payload_str) - Busca dados usando payload JSON
Mostra confirmação das funções carregadas -->

In [3]:
# ==========================================
# FUNÇÕES DE API
# ==========================================

def listar_catalogos(verbose: bool = False) -> pd.DataFrame:
    """
    Lista todos os catálogos disponíveis na API.
    
    Args:
        verbose: Se True, exibe mensagens informativas
    
    Returns:
        DataFrame com os catálogos disponíveis (codCatalogo, nome, descricao, etc.)
    """
    url = f"{BASE_API_URL}/api/Catalogo/ListarCatalogos"
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    
    if "data" in data and isinstance(data["data"], list):
        df = pd.DataFrame(data["data"])
        if verbose:
            print(f"✅ Encontrados {len(df)} catálogos disponíveis")
        return df
    else:
        if verbose:
            print("⚠️  Resposta da API não contém lista de catálogos")
        return pd.DataFrame()


def obter_dados_etapa_atual(TOKEN: str, codigo_catalogo: list, verbose: bool = False) -> pd.DataFrame:
    """
    Obtém dados da etapa atual do relatório de etapas.
    
    Args:
        TOKEN: Token de autenticação
        codigo_catalogo: Lista com códigos de catálogo a consultar
        verbose: Se True, exibe mensagens informativas
    
    Returns:
        DataFrame com os dados das etapas
    """
    url = f"{BASE_API_URL}/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"
    
    headers_atualizados = HEADERS.copy()
    headers_atualizados["Authorization"] = f"Bearer {TOKEN}"
    
    def processar_resposta(data, verbose_inner=False):
        """Processa resposta da API extraindo lista de dados."""
        if isinstance(data, list):
            return pd.DataFrame(data), data
        elif isinstance(data, dict):
            for key in ["data", "result", "items"]:
                if key in data and isinstance(data[key], list):
                    return pd.DataFrame(data[key]), data
            if verbose_inner:
                print(f"⚠️  Nenhuma chave conhecida encontrada na resposta")
            return pd.DataFrame(), data
        else:
            if verbose_inner:
                print(f"⚠️  Tipo não esperado na resposta: {type(data)}")
            return pd.DataFrame(), data
    
    payload_etapa = {
        "codCatalogos": codigo_catalogo,
        "dataInicio": "2020-01-01T00:00:00.000Z",
        "dataFim": HOJE,
        "ativo": 1,
    }
    
    try:
        r = requests.post(url, json=payload_etapa, headers=headers_atualizados, timeout=120)
        r.raise_for_status()
        data = r.json()
        
        df, _ = processar_resposta(data, verbose_inner=verbose)
        
        if df.empty and verbose:
            print("⚠️  DataFrame vazio após processamento")
        elif verbose:
            print(f"✅ DataFrame criado: {len(df)} linhas, {len(df.columns)} colunas")
        
        return df
        
    except requests.exceptions.HTTPError as e:
        if r.status_code == 401:
            error_msg = (
                "❌ Erro 401: Token não autorizado ou expirado!\n"
                "   Verifique se o TOKEN_SANTOS_OBRAS está atualizado no config_api_acto.ipynb"
            )
            if verbose:
                print(error_msg)
            raise ValueError(error_msg) from e
        raise
    except Exception as e:
        if verbose:
            print(f"❌ Erro ao processar resposta da API: {e}")
        raise


def fetch_tabela(payload_str: str, verbose: bool = False) -> pd.DataFrame:
    """
    Recebe o JSON do payload como string e devolve um DataFrame com os dados.
    
    Este payload geralmente é copiado do DevTools quando você visualiza uma tabela
    na aplicação web. Para obter:
    1. Abra DevTools → Network
    2. Filtre por 'VisualizarDadosIntermediarios'
    3. Clique na requisição → Aba Payload
    4. Copie o JSON completo
    
    Args:
        payload_str: String JSON com a configuração da tabela (ou dict já parseado)
        verbose: Se True, exibe mensagens informativas
    
    Returns:
        DataFrame com os dados da tabela
    """
    url_dados = f"{BASE_API_URL}/api/Tabela/VisualizarDadosIntermediarios"
    
    config = json.loads(payload_str)
    resp = requests.post(url_dados, headers=HEADERS, json=config)
    print("Status:", resp.status_code)
    
    if resp.status_code != 200:
        print("Corpo da resposta:", resp.text[:500])
        raise SystemExit("❌ Erro ao chamar API")
    
    data = resp.json()
    
    lista_final = []
    if "data" in data and isinstance(data["data"], list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for key, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)
    if not lista_final:
        print("✅ Requisição OK, mas sem linhas.")
        return pd.DataFrame()
    
    df = pd.DataFrame(lista_final)
    print(f"✅ Linhas em pandas: {len(df)}")
    return df
    
    df = pd.DataFrame(lista_final)
    
    if verbose:
        print(f"✅ DataFrame criado: {len(df)} linhas, {len(df.columns)} colunas")
    
    return df

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 5, Finished, Available, Finished)

In [4]:
# ==========================================
# PAYLOAD JSON E EXTRAÇÃO DE CÓDIGOS
# ==========================================

payload_json_completo = """{"nome":"Base_obras_santos","solicitacoes":[[{"codCatalogo":4803,"codConfigColCatalogo":0,"nomeServico":"INSCRIÇÃO DE PROFISSIONAL (PESSOA FÍSICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":4803,"etapasDados":{"nomeServico":"4803","etapas":[10641,16412,10643]}}]},"filtros":null,"servicos":[{"codConfigCol":null,"col":"seqFluxo","tit":"Nº Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":1,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"servico","tit":"Serviço","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":2,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null}],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":4804,"codConfigColCatalogo":0,"nomeServico":"INSCRIÇÃO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":4804,"etapasDados":{"nomeServico":"4804","etapas":[10616,10622,16592]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5605,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DA LICENÇA DE OPERAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5605,"etapasDados":{"nomeServico":"5605","etapas":[14446,14439,14445,14453]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5625,"codConfigColCatalogo":0,"nomeServico":"LICENÇA DE INSTALAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5625,"etapasDados":{"nomeServico":"5625","etapas":[14491,14490,14484]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5626,"codConfigColCatalogo":0,"nomeServico":"LICENÇA DE OPERAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5626,"etapasDados":{"nomeServico":"5626","etapas":[14467,14468,14475,14461]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5627,"codConfigColCatalogo":0,"nomeServico":"LICENÇA PRÉVIA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5627,"etapasDados":{"nomeServico":"5627","etapas":[14430,14421,14415,14422]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5628,"codConfigColCatalogo":0,"nomeServico":"MANIFESTAÇÃO TÉCNICA AMBIENTAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5628,"etapasDados":{"nomeServico":"5628","etapas":[14538,14544,14535,14528]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5677,"codConfigColCatalogo":0,"nomeServico":"COMUNICAÇÃO DE SERVIÇOS ISENTOS DE LICENÇA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5677,"etapasDados":{"nomeServico":"5677","etapas":[22456]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5679,"codConfigColCatalogo":0,"nomeServico":"CONSTRUÇÃO NOVAS DE EDIFICAÇÕES -SOBREPOSTA E/OU G","etapasSelecionadas":{"catalogo":[{"codCatalogo":5679,"etapasDados":{"nomeServico":"5679","etapas":[14754,14763,14764,14733,14762,14761]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5685,"codConfigColCatalogo":0,"nomeServico":"COMUNICAÇÃO DE INÍCIO DE OBRAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":5685,"etapasDados":{"nomeServico":"5685","etapas":[18345]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5686,"codConfigColCatalogo":0,"nomeServico":"CONSTRUÇÃO NOVAS DE EDIFICAÇÕES - UNIFAMILIAR","etapasSelecionadas":{"catalogo":[{"codCatalogo":5686,"etapasDados":{"nomeServico":"5686","etapas":[14938,14934,14932,14940,14931,14937,14939]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5693,"codConfigColCatalogo":0,"nomeServico":"DEMOLIÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5693,"etapasDados":{"nomeServico":"5693","etapas":[17581,17575,17583,17584,17582,17579]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5725,"codConfigColCatalogo":0,"nomeServico":"ALVARÁ DE CONSTRUÇÃO PLURI-HABITACIONAL VERTICAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5725,"etapasDados":{"nomeServico":"5725","etapas":[14985,14996,14984,14987,14993,14994,14995]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5755,"codConfigColCatalogo":0,"nomeServico":"ALVARÁ DE CONSTRUÇÃO NOVA CONDOMÍNIO HORIZONTAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5755,"etapasDados":{"nomeServico":"5755","etapas":[15263,15264,15265,15253,15262,15252,15255]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5964,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO PROFISSIONAL PESSOA FÍSICA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5964,"etapasDados":{"nomeServico":"5964","etapas":[16282,16283]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6093,"codConfigColCatalogo":0,"nomeServico":"HABITE-SE","etapasSelecionadas":{"catalogo":[{"codCatalogo":6093,"etapasDados":{"nomeServico":"6093","etapas":[16811,16815,16813]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6113,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":6113,"etapasDados":{"nomeServico":"6113","etapas":[17033,17034,17032]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6326,"codConfigColCatalogo":0,"nomeServico":"PROJETO URBANÍSTICO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6326,"etapasDados":{"nomeServico":"6326","etapas":[18360,18361,18359,18356,18353]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6383,"codConfigColCatalogo":0,"nomeServico":"ACOMPANHAMENTO DE OBRAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":6383,"etapasDados":{"nomeServico":"6383","etapas":[24356,18548,18554]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6513,"codConfigColCatalogo":0,"nomeServico":"NOVAS EDIFICAÇÕES COMERCIAL, SERVIÇOS E/OU MISTO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6513,"etapasDados":{"nomeServico":"6513","etapas":[18014,18022,18023,18025,18013,18024,18016]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6738,"codConfigColCatalogo":0,"nomeServico":"CADASTRO DE EMPRESAS DE INSTALAÇÃO E MANUTENÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6738,"etapasDados":{"nomeServico":"6738","etapas":[19761,19775,20152,19764]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6783,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO  DE EMPRESAS DE INSTALAÇÃO E MANUTENÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6783,"etapasDados":{"nomeServico":"6783","etapas":[20154,19873]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6963,"codConfigColCatalogo":0,"nomeServico":"ADMINISTRAÇÃO FICHA ROSA","etapasSelecionadas":{"catalogo":[{"codCatalogo":6963,"etapasDados":{"nomeServico":"6963","etapas":[20412]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":7523,"codConfigColCatalogo":0,"nomeServico":"ASSUNÇÃO DE RESP. TÉCNICA EQUIPAMENTOS","etapasSelecionadas":{"catalogo":[{"codCatalogo":7523,"etapasDados":{"nomeServico":"7523","etapas":[23033,23035]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":8134,"codConfigColCatalogo":0,"nomeServico":"PROVIDÊNCIA","etapasSelecionadas":{"catalogo":[{"codCatalogo":8134,"etapasDados":{"nomeServico":"8134","etapas":[25413]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":12804,"codConfigColCatalogo":0,"nomeServico":"MANUTENÇÃO DE FACHADAS EM EDIFICAÇÕES HISTÓRICAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":12804,"etapasDados":{"nomeServico":"12804","etapas":[39156]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null}]],"ativo":true,"dataCriacao":"2026-01-16T15:39:40.885Z","dateDataAlteracao":"2026-01-16T15:39:40.885Z","filtros":[],"campos":[],"filtroData":false,"parametrosFiltrosData":[],"id":"696a5bbc1c7f6221017ae73f","dateDataAlteracaoFormatada":"16/01/2026"}"""


def validar_payload(payload_str: str) -> dict:
    """
    Valida e retorna informações sobre o payload JSON.
    
    Args:
        payload_str: String JSON com a configuração da tabela
        
    Returns:
        Dicionário com informações de validação
    """
    try:
        config = json.loads(payload_str)
        info = {
            "valido": True,
            "nome": config.get("nome", ""),
            "id": config.get("id", ""),
            "ativo": config.get("ativo", False),
            "total_solicitacoes": 0,
            "total_cod_catalogo": 0,
            "cod_catalogos": []
        }
        
        if "solicitacoes" in config and len(config["solicitacoes"]) > 0:
            info["total_solicitacoes"] = len(config["solicitacoes"])
            codigos = []
            for grupo in config["solicitacoes"]:
                if isinstance(grupo, list):
                    for item in grupo:
                        if isinstance(item, dict) and "codCatalogo" in item:
                            cod = item["codCatalogo"]
                            if cod and cod not in codigos:
                                codigos.append(cod)
            info["total_cod_catalogo"] = len(codigos)
            info["cod_catalogos"] = sorted(codigos)
        
        return info
    except Exception as e:
        return {
            "valido": False,
            "erro": str(e)
        }


def extrair_cod_catalogo(payload_str: str) -> list:
    """
    Extrai todos os codCatalogo únicos do payload JSON.
    
    Args:
        payload_str: String JSON com a configuração da tabela
        
    Returns:
        Lista com todos os códigos de catálogo encontrados
    """
    try:
        config = json.loads(payload_str)
        codigos = []
        
        if "solicitacoes" in config and len(config["solicitacoes"]) > 0:
            for grupo in config["solicitacoes"]:
                if isinstance(grupo, list):
                    for item in grupo:
                        if isinstance(item, dict) and "codCatalogo" in item:
                            cod = item["codCatalogo"]
                            if cod and cod not in codigos:
                                codigos.append(cod)
        
        if codigos:
            print(f"✅ Encontrados {len(codigos)} códigos de catálogo")
            return codigos
        else:
            print("⚠️  Nenhum codCatalogo encontrado no payload")
            return []
    except Exception as e:
        print(f"❌ Erro ao processar payload: {e}")
        return []

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 6, Finished, Available, Finished)

# Funções tb_os_acto

In [5]:
# ==========================================
# PAYLOAD JSON E EXTRAÇÃO DE CÓDIGOS
# ==========================================

payload_json_completo = """{"nome":"Base_obras_santos","solicitacoes":[[{"codCatalogo":4803,"codConfigColCatalogo":0,"nomeServico":"INSCRIÇÃO DE PROFISSIONAL (PESSOA FÍSICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":4803,"etapasDados":{"nomeServico":"4803","etapas":[10641,16412,10643]}}]},"filtros":null,"servicos":[{"codConfigCol":null,"col":"seqFluxo","tit":"Nº Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":1,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"servico","tit":"Serviço","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":2,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null}],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":4804,"codConfigColCatalogo":0,"nomeServico":"INSCRIÇÃO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":4804,"etapasDados":{"nomeServico":"4804","etapas":[10616,10622,16592]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5605,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DA LICENÇA DE OPERAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5605,"etapasDados":{"nomeServico":"5605","etapas":[14446,14439,14445,14453]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5625,"codConfigColCatalogo":0,"nomeServico":"LICENÇA DE INSTALAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5625,"etapasDados":{"nomeServico":"5625","etapas":[14491,14490,14484]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5626,"codConfigColCatalogo":0,"nomeServico":"LICENÇA DE OPERAÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5626,"etapasDados":{"nomeServico":"5626","etapas":[14467,14468,14475,14461]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5627,"codConfigColCatalogo":0,"nomeServico":"LICENÇA PRÉVIA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5627,"etapasDados":{"nomeServico":"5627","etapas":[14430,14421,14415,14422]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5628,"codConfigColCatalogo":0,"nomeServico":"MANIFESTAÇÃO TÉCNICA AMBIENTAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5628,"etapasDados":{"nomeServico":"5628","etapas":[14538,14544,14535,14528]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5677,"codConfigColCatalogo":0,"nomeServico":"COMUNICAÇÃO DE SERVIÇOS ISENTOS DE LICENÇA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5677,"etapasDados":{"nomeServico":"5677","etapas":[22456]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5679,"codConfigColCatalogo":0,"nomeServico":"CONSTRUÇÃO NOVAS DE EDIFICAÇÕES -SOBREPOSTA E/OU G","etapasSelecionadas":{"catalogo":[{"codCatalogo":5679,"etapasDados":{"nomeServico":"5679","etapas":[14754,14763,14764,14733,14762,14761]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5685,"codConfigColCatalogo":0,"nomeServico":"COMUNICAÇÃO DE INÍCIO DE OBRAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":5685,"etapasDados":{"nomeServico":"5685","etapas":[18345]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5686,"codConfigColCatalogo":0,"nomeServico":"CONSTRUÇÃO NOVAS DE EDIFICAÇÕES - UNIFAMILIAR","etapasSelecionadas":{"catalogo":[{"codCatalogo":5686,"etapasDados":{"nomeServico":"5686","etapas":[14938,14934,14932,14940,14931,14937,14939]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5693,"codConfigColCatalogo":0,"nomeServico":"DEMOLIÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":5693,"etapasDados":{"nomeServico":"5693","etapas":[17581,17575,17583,17584,17582,17579]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5725,"codConfigColCatalogo":0,"nomeServico":"ALVARÁ DE CONSTRUÇÃO PLURI-HABITACIONAL VERTICAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5725,"etapasDados":{"nomeServico":"5725","etapas":[14985,14996,14984,14987,14993,14994,14995]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5755,"codConfigColCatalogo":0,"nomeServico":"ALVARÁ DE CONSTRUÇÃO NOVA CONDOMÍNIO HORIZONTAL","etapasSelecionadas":{"catalogo":[{"codCatalogo":5755,"etapasDados":{"nomeServico":"5755","etapas":[15263,15264,15265,15253,15262,15252,15255]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":5964,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO PROFISSIONAL PESSOA FÍSICA","etapasSelecionadas":{"catalogo":[{"codCatalogo":5964,"etapasDados":{"nomeServico":"5964","etapas":[16282,16283]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6093,"codConfigColCatalogo":0,"nomeServico":"HABITE-SE","etapasSelecionadas":{"catalogo":[{"codCatalogo":6093,"etapasDados":{"nomeServico":"6093","etapas":[16811,16815,16813]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6113,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO EMPRESA/PROFISSIONAL (PESSOA JURÍDICA)","etapasSelecionadas":{"catalogo":[{"codCatalogo":6113,"etapasDados":{"nomeServico":"6113","etapas":[17033,17034,17032]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6326,"codConfigColCatalogo":0,"nomeServico":"PROJETO URBANÍSTICO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6326,"etapasDados":{"nomeServico":"6326","etapas":[18360,18361,18359,18356,18353]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6383,"codConfigColCatalogo":0,"nomeServico":"ACOMPANHAMENTO DE OBRAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":6383,"etapasDados":{"nomeServico":"6383","etapas":[24356,18548,18554]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6513,"codConfigColCatalogo":0,"nomeServico":"NOVAS EDIFICAÇÕES COMERCIAL, SERVIÇOS E/OU MISTO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6513,"etapasDados":{"nomeServico":"6513","etapas":[18014,18022,18023,18025,18013,18024,18016]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6738,"codConfigColCatalogo":0,"nomeServico":"CADASTRO DE EMPRESAS DE INSTALAÇÃO E MANUTENÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6738,"etapasDados":{"nomeServico":"6738","etapas":[19761,19775,20152,19764]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6783,"codConfigColCatalogo":0,"nomeServico":"RENOVAÇÃO DE CADASTRO  DE EMPRESAS DE INSTALAÇÃO E MANUTENÇÃO","etapasSelecionadas":{"catalogo":[{"codCatalogo":6783,"etapasDados":{"nomeServico":"6783","etapas":[20154,19873]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":6963,"codConfigColCatalogo":0,"nomeServico":"ADMINISTRAÇÃO FICHA ROSA","etapasSelecionadas":{"catalogo":[{"codCatalogo":6963,"etapasDados":{"nomeServico":"6963","etapas":[20412]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":7523,"codConfigColCatalogo":0,"nomeServico":"ASSUNÇÃO DE RESP. TÉCNICA EQUIPAMENTOS","etapasSelecionadas":{"catalogo":[{"codCatalogo":7523,"etapasDados":{"nomeServico":"7523","etapas":[23033,23035]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":8134,"codConfigColCatalogo":0,"nomeServico":"PROVIDÊNCIA","etapasSelecionadas":{"catalogo":[{"codCatalogo":8134,"etapasDados":{"nomeServico":"8134","etapas":[25413]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":12804,"codConfigColCatalogo":0,"nomeServico":"MANUTENÇÃO DE FACHADAS EM EDIFICAÇÕES HISTÓRICAS","etapasSelecionadas":{"catalogo":[{"codCatalogo":12804,"etapasDados":{"nomeServico":"12804","etapas":[39156]}}]},"filtros":null,"servicos":[],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null}]],"ativo":true,"dataCriacao":"2026-01-16T15:39:40.885Z","dateDataAlteracao":"2026-01-16T15:39:40.885Z","filtros":[],"campos":[],"filtroData":false,"parametrosFiltrosData":[],"id":"696a5bbc1c7f6221017ae73f","dateDataAlteracaoFormatada":"16/01/2026"}"""


def validar_payload(payload_str: str) -> dict:
    """
    Valida e retorna informações sobre o payload JSON.
    
    Args:
        payload_str: String JSON com a configuração da tabela (ou dict já parseado)
        
    Returns:
        Dicionário com informações de validação
    """
    try:
        config = json.loads(payload_str) if isinstance(payload_str, str) else payload_str
        info = {
            "valido": True,
            "nome": config.get("nome", ""),
            "id": config.get("id", ""),
            "ativo": config.get("ativo", False),
            "total_solicitacoes": 0,
            "total_cod_catalogo": 0,
            "cod_catalogos": []
        }
        
        if "solicitacoes" in config and len(config["solicitacoes"]) > 0:
            info["total_solicitacoes"] = len(config["solicitacoes"])
            codigos = []
            for grupo in config["solicitacoes"]:
                if isinstance(grupo, list):
                    for item in grupo:
                        if isinstance(item, dict) and "codCatalogo" in item:
                            cod = item["codCatalogo"]
                            if cod and cod not in codigos:
                                codigos.append(cod)
            info["total_cod_catalogo"] = len(codigos)
            info["cod_catalogos"] = sorted(codigos)
        
        return info
    except Exception as e:
        return {
            "valido": False,
            "erro": str(e)
        }


def extrair_cod_catalogo(payload_str: str, verbose: bool = False) -> list:
    """
    Extrai todos os codCatalogo únicos do payload JSON.
    
    Args:
        payload_str: String JSON com a configuração da tabela (ou dict já parseado)
        verbose: Se True, exibe mensagens informativas
        
    Returns:
        Lista com todos os códigos de catálogo encontrados
    """
    try:
        config = json.loads(payload_str) if isinstance(payload_str, str) else payload_str
        codigos = []
        
        if "solicitacoes" in config and len(config["solicitacoes"]) > 0:
            for grupo in config["solicitacoes"]:
                if isinstance(grupo, list):
                    for item in grupo:
                        if isinstance(item, dict) and "codCatalogo" in item:
                            cod = item["codCatalogo"]
                            if cod and cod not in codigos:
                                codigos.append(cod)
        
        if codigos and verbose:
            print(f"✅ Encontrados {len(codigos)} códigos de catálogo")
        elif not codigos and verbose:
            print("⚠️  Nenhum codCatalogo encontrado no payload")
        
        return codigos
    except Exception as e:
        if verbose:
            print(f"❌ Erro ao processar payload: {e}")
        return []

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 7, Finished, Available, Finished)

In [6]:
# ==========================================
# FUNÇÕES UTILITÁRIAS - ETAPAS E SOLICITAÇÕES
# ==========================================

def adicionar_etapa_atual_2(
    df_etapas: pd.DataFrame, df_solicitacoes: pd.DataFrame
) -> pd.DataFrame:
    """
    Adiciona etapa atual às solicitações usando coluna 'Nº Solicitação'.
    
    Args:
        df_etapas: DataFrame com dados de etapas
        df_solicitacoes: DataFrame com dados de solicitações
    
    Returns:
        DataFrame de solicitações com colunas etapa e executor adicionadas
    """
    temp = df_etapas.groupby("seqFluxo", as_index=False)["dataAtenderEtapa"].max()
    temp_etapa_os = (
        df_etapas.merge(temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner")
        .groupby("seqFluxo", as_index=False)
        .size()
    )
    lista = (
        temp_etapa_os.sort_values("size", ascending=False)
        .query("size > 1")["seqFluxo"]
        .tolist()
    )
    temp = temp.loc[~temp["seqFluxo"].isin(lista)].copy()
    df_etapa_atual = df_etapas.merge(
        temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner"
    )
    df_etapa_comeca_mesmo_tempo = df_etapas.query(
        "seqFluxo in @lista and dataEtapaFim.isna()"
    ).sort_values(["seqFluxo", "dataAtenderEtapa"])
    df_etapa_atual2 = pd.concat([df_etapa_atual, df_etapa_comeca_mesmo_tempo], axis=0)
    df_etapa_atual2 = df_etapa_atual2[["seqFluxo", "etapa", "executor"]].copy()
    df_solicitacoes = df_solicitacoes.rename(columns={"Nº Solicitação": "seqFluxo"})
    df_solicitacoes = df_solicitacoes.astype({"seqFluxo": "int64"})
    df_solicitacoes["Data Criação"] = pd.to_datetime(
        df_solicitacoes["Data Criação"], format="ISO8601"
    )
    df_solicitacoes = df_solicitacoes.merge(df_etapa_atual2, on="seqFluxo", how="left")

    return df_solicitacoes


def criar_tabela_solicitacoes_completa(
    payload_str: str = None,
    cod_catalogo: list = None,
    adicionar_etapa_atual: bool = True,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Cria tabela completa de solicitações com todas as etapas e tratamentos.
    
    Esta função integra todas as etapas necessárias para criar a tabela de solicitações:
    1. Extrai códigos de catálogo do payload (se não fornecidos)
    2. Busca dados de etapas da API
    3. Busca dados de solicitações usando o payload
    4. Adiciona etapa atual às solicitações
    5. Aplica tratamentos de dados
    
    Args:
        payload_str: String JSON com a configuração da tabela (usa payload_json_completo se None)
        cod_catalogo: Lista de códigos de catálogo (extrai do payload se None)
        adicionar_etapa_atual: Se True, adiciona etapa atual às solicitações
        verbose: Se True, exibe mensagens informativas
    
    Returns:
        DataFrame com tabela de solicitações completa e tratada
    """
    if payload_str is None:
        payload_str = payload_json_completo
    
    if verbose:
        print("🔍 Iniciando criação da tabela de solicitações...")
    
    # 1. Extrair códigos de catálogo se não fornecidos
    if cod_catalogo is None:
        if verbose:
            print("📋 Extraindo códigos de catálogo do payload...")
        cod_catalogo = extrair_cod_catalogo(payload_str, verbose=verbose)
        if not cod_catalogo:
            raise ValueError("Nenhum código de catálogo encontrado no payload")
    
    # 2. Buscar dados de etapas
    if verbose:
        print(f"📥 Buscando dados de etapas para {len(cod_catalogo)} catálogos...")
    df_etapas = obter_dados_etapa_atual(TOKEN, cod_catalogo, verbose=verbose)
    
    # 3. Buscar dados de solicitações
    if verbose:
        print("📥 Buscando dados de solicitações...")
    df_solicitacoes = fetch_tabela(payload_str, verbose=verbose)
    
    if df_solicitacoes.empty:
        if verbose:
            print("⚠️  Nenhuma solicitação encontrada")
        return df_solicitacoes
    
    # 4. Adicionar etapa atual se solicitado
    if adicionar_etapa_atual and not df_etapas.empty:
        if verbose:
            print("🔗 Adicionando etapa atual às solicitações...")
        df_solicitacoes = adicionar_etapa_atual_2(df_etapas, df_solicitacoes)
    
    if verbose:
        print(f"✅ Tabela de solicitações criada: {len(df_solicitacoes)} registros")
        print(f"   Colunas: {list(df_solicitacoes.columns)}")
    
    return df_solicitacoes

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 8, Finished, Available, Finished)

In [7]:
# ==========================================
# FUNÇÕES UTILITÁRIAS - TRATAMENTO DE DADOS
# ==========================================

def aplicar_bfill(df_solicitacoes_obras, coluna: str):
    n_solicitacao = df_solicitacoes_obras.filter(like=coluna).columns
    df_solicitacoes_obras[coluna] = (
        df_solicitacoes_obras[n_solicitacao].bfill(axis=1).iloc[:, 0]
    )
    df_solicitacoes_obras = df_solicitacoes_obras.drop(columns=n_solicitacao)
    return df_solicitacoes_obras


def harmonizar_nome_bairros(df):
    """
    Harmoniza nomes de bairros para padronização.
    
    Aplica substituições para corresponder ao shapefile do Power BI.
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    
    A função aplica title() primeiro e depois as substituições.
    """
    if 'bairro_consolidado' not in df.columns:
        # Se não tem bairro_consolidado, criar a partir de bairro
        if 'bairro' in df.columns:
            df['bairro_consolidado'] = df['bairro'].str.title().str.strip()
        else:
            return df
    
    # Lista de substituições (alinhada com processo manual)
    substituicoes_bairro = [
        # Correções de acentuação e grafia
        ("Radio", "Rádio"),
        ("Boqueirao", "Boqueirão"),
        ("Pompeia", "Pompéia"),
        ("Ilheu", "Ilhéu"),
        ("Jose", "José"),
        ("Embare", "Embaré"),
        ("Ponta Da Praia", "Ponta da Praia"),
        ("Porta Da Praia", "Ponta da Praia"),
        ("Porto Da Praia", "Ponta da Praia"),
        ("Marape", "Marapé"),
        ("Vila Matias", "Vila Mathias"),
        # Correções de Morro
        ("Mr.", "Morro"),
        ("Mor.", "Morro"),
        ("Morro Monte Serrat", "Monte Serrat"),
        ("Morro Jose Menino", "Morro José Menino"),
        ("Morro Da Caneleira", "Morro Caneleira"),
        ("Morro Caneleira", "Morro Caneleira"),
        # Outras correções
        ("Chico De Paula", "Chico de Paula"),
        ("Iriri Alto", "Iriri"),
        ("Ilha Barnabé", "Barnabe"),
    ]
    
    # Aplicar substituições
    for old, new in substituicoes_bairro:
        df['bairro_consolidado'] = df['bairro_consolidado'].str.replace(old, new, regex=False)
    
    return df


def ajustar_nome_colunas(df):
    """
    Padroniza nomes de colunas para lowercase e remove acentos/símbolos.
    """
    df.columns = (
        df.columns.str.lower()
        .str.strip()
        .str.replace("  ", " ")
        .str.replace("º", "")
        .str.replace(":", "")
        .str.replace("ç", "c")
        .str.replace("ã", "a")
        .str.replace("ú", "u")
        .str.replace("ê", "e")
        .str.replace("á", "a")
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 9, Finished, Available, Finished)

In [8]:
# ==========================================
# FUNÇÕES DE TRATAMENTO - ARQUIVO AUXILIAR
# ==========================================

ARQUIVO_AUX = "/lakehouse/default/Files/acto/PMS_AuxiliarPDR.xlsx"


def processar_prazo():
    """
    Processa tabela auxiliar de prazos.
    
    Tenta ler da aba 'aux_prazo'. Se não existir, tenta outras abas possíveis.
    A aba deve conter colunas: 'servico' (ou 'Serviço') e 'prazo_de_conclusao' (ou similar).
    """
    try:
        # Tentar ler da aba aux_prazo primeiro
        prazo = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_prazo")
    except (ValueError, FileNotFoundError):
        try:
            # Tentar ler da aba aux_Geral (pode ter estrutura diferente)
            prazo = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_Geral")
            # Se aux_Geral não tiver prazo, criar estrutura vazia
            if "prazo_de_conclusao" not in [c.lower() for c in prazo.columns]:
                print("⚠️  Aba aux_Geral não contém informações de prazo.")
                print("   Criando estrutura vazia. Prazos não serão aplicados.")
                # Criar DataFrame vazio com estrutura esperada
                prazo = pd.DataFrame(columns=["servico", "prazo_de_conclusao"])
                return prazo
        except Exception as e:
            print(f"⚠️  Erro ao ler aba de prazos: {e}")
            print("   Criando estrutura vazia. Prazos não serão aplicados.")
            prazo = pd.DataFrame(columns=["servico", "prazo_de_conclusao"])
            return prazo
    
    # Ajustar nomes de colunas
    prazo = ajustar_nome_colunas(prazo)
    
    # Verificar se tem coluna de serviço
    if "servico" not in prazo.columns:
        # Tentar encontrar coluna similar
        servico_cols = [c for c in prazo.columns if "servi" in c.lower()]
        if servico_cols:
            prazo = prazo.rename(columns={servico_cols[0]: "servico"})
        else:
            print("⚠️  Coluna 'servico' não encontrada na aba de prazos.")
            prazo = pd.DataFrame(columns=["servico", "prazo_de_conclusao"])
            return prazo
    
    # Verificar se tem coluna de prazo
    if "prazo_de_conclusao" not in prazo.columns:
        # Tentar encontrar coluna similar
        prazo_cols = [c for c in prazo.columns if any(x in c.lower() for x in ["prazo", "dias", "tempo", "deadline"])]
        if prazo_cols:
            prazo = prazo.rename(columns={prazo_cols[0]: "prazo_de_conclusao"})
        else:
            print("⚠️  Coluna 'prazo_de_conclusao' não encontrada na aba de prazos.")
            print("   Prazos não serão aplicados.")
            # Criar coluna vazia para não quebrar o merge
            prazo["prazo_de_conclusao"] = None
    
    # Normalizar serviço
    if prazo["servico"].dtype == "object":
        prazo["servico"] = prazo["servico"].str.capitalize()
    
    return prazo


def processar_bairros():
    """
    Processa tabela auxiliar de bairros e regionais.
    
    Lê da aba 'Zona_Bairros' que contém informações de bairros e zonas.
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    """
    try:
        # Ler da aba Zona_Bairros
        bairros = pd.read_excel(ARQUIVO_AUX, sheet_name="Zona_Bairros")
    except (ValueError, FileNotFoundError):
        try:
            # Tentar aba alternativa
            bairros = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_regionais")
        except Exception as e:
            print(f"⚠️  Erro ao ler aba de bairros: {e}")
            print("   Criando estrutura vazia. Bairros não serão aplicados.")
            bairros = pd.DataFrame(columns=["bairro_consolidado", "zona", "regiao"])
            return bairros
    
    # Processamento alinhado com processo manual:
    # 1. Remover nulos em BAIRRO
    if "BAIRRO" in bairros.columns:
        bairros = bairros.dropna(subset=["BAIRRO"])
    
    # 2. Padronizar ZONA antes de ajustar nomes
    if "ZONA" in bairros.columns:
        bairros["ZONA"] = bairros["ZONA"].astype(str).str.strip()
    
    # 3. Ajustar nomes de colunas (lowercase, remove acentos)
    bairros = ajustar_nome_colunas(bairros)
    
    # 4. Verificar e mapear colunas
    if "bairro" not in bairros.columns:
        bairro_cols = [c for c in bairros.columns if "bairro" in c.lower()]
        if bairro_cols:
            bairros = bairros.rename(columns={bairro_cols[0]: "bairro"})
        else:
            print("⚠️  Coluna 'bairro' não encontrada na aba de bairros.")
            bairros = pd.DataFrame(columns=["bairro_consolidado", "zona", "regiao"])
            return bairros
    
    # 5. Criar coluna bairro_consolidado (title case para corresponder ao shapefile)
    # Primeiro padroniza para title case, depois aplica harmonização
    bairros["bairro_consolidado"] = bairros["bairro"].astype(str).str.title().str.strip()
    
    # 7. Manter apenas BAIRRO e ZONA (como no processo manual)
    colunas_manter = ["bairro_consolidado"]
    if "zona" in bairros.columns:
        colunas_manter.append("zona")
    if "regiao" in bairros.columns:
        colunas_manter.append("regiao")
    elif "regional" in bairros.columns:
        bairros = bairros.rename(columns={"regional": "regiao"})
        colunas_manter.append("regiao")
    if "fiscal" in bairros.columns:
        colunas_manter.append("fiscal")
    
    # 8. Remover duplicatas (como no processo manual)
    bairros = bairros[colunas_manter].drop_duplicates()
    
    return bairros


def processar_etapas():
    """
    Processa tabela auxiliar de etapas.
    
    Lê da aba 'Etapas' que contém mapeamento etapa → setor responsável, PDR, zona.
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    """
    try:
        # Ler da aba Etapas
        etapas = pd.read_excel(ARQUIVO_AUX, sheet_name="Etapas")
    except (ValueError, FileNotFoundError) as e:
        print(f"⚠️  Erro ao ler aba Etapas: {e}")
        print("   Criando estrutura vazia. Etapas não serão aplicadas.")
        etapas = pd.DataFrame(columns=["etapa", "aux_setor_responsavel", "aux_pdr", "zona"])
        return etapas
    
    # Processamento alinhado com processo manual:
    # 1. Padronizar Etapa (uppercase, strip)
    if "Etapa" in etapas.columns:
        etapas["Etapa"] = etapas["Etapa"].astype(str).str.upper().str.strip()
    
    # 2. Ajustar nomes de colunas
    etapas = ajustar_nome_colunas(etapas)
    
    # 3. Renomear colunas para padrão esperado
    rename_map = {
        "etapa": "etapa_aux",
        "auxsetorresponsavel": "aux_setor_responsavel",
        "auxpdr": "aux_pdr",
    }
    
    for old_col, new_col in rename_map.items():
        if old_col in etapas.columns:
            etapas = etapas.rename(columns={old_col: new_col})
    
    # 4. Verificar se tem coluna zona
    if "zona" not in etapas.columns:
        # Tentar encontrar coluna similar
        zona_cols = [c for c in etapas.columns if "zona" in c.lower()]
        if zona_cols:
            etapas = etapas.rename(columns={zona_cols[0]: "zona_etapa"})
        else:
            etapas["zona_etapa"] = None
    
    # 5. Manter apenas colunas relevantes
    colunas_manter = ["etapa_aux", "aux_setor_responsavel", "aux_pdr"]
    if "zona_etapa" in etapas.columns:
        colunas_manter.append("zona_etapa")
    
    etapas = etapas[[c for c in colunas_manter if c in etapas.columns]]
    
    return etapas


def aplicar_merge_etapas(df, coluna_etapa="etapa_atual"):
    """
    Faz merge com tabela auxiliar de etapas.
    
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    
    Args:
        df: DataFrame principal
        coluna_etapa: Nome da coluna que contém a etapa (padrão: "etapa_atual")
    
    Returns:
        DataFrame com colunas adicionais: aux_setor_responsavel, aux_pdr, zona_etapa
    """
    etapas = processar_etapas()
    
    if etapas.empty:
        print("⚠️  Tabela de etapas vazia. Merge não será realizado.")
        return df
    
    # Criar coluna padronizada para merge
    df_merged = df.copy()
    df_merged["etapa_pad"] = df_merged[coluna_etapa].astype(str).str.upper().str.strip()
    
    # Fazer merge
    df_merged = df_merged.merge(
        etapas,
        left_on="etapa_pad",
        right_on="etapa_aux",
        how="left"
    )
    
    # Remover colunas auxiliares
    if "etapa_pad" in df_merged.columns:
        df_merged = df_merged.drop(columns=["etapa_pad"])
    if "etapa_aux" in df_merged.columns:
        df_merged = df_merged.drop(columns=["etapa_aux"])
    
    return df_merged


def aplicar_merge_bairros(df, coluna_bairro="bairro"):
    """
    Faz merge com tabela auxiliar de bairros e zonas.
    
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    
    Args:
        df: DataFrame principal
        coluna_bairro: Nome da coluna que contém o bairro (padrão: "bairro")
    
    Returns:
        DataFrame com colunas adicionais: zona_bairro (ou zona se consolidada)
    """
    bairros = processar_bairros()
    
    if bairros.empty:
        print("⚠️  Tabela de bairros vazia. Merge não será realizado.")
        return df
    
    # Criar coluna padronizada para merge (uppercase, strip, remove espaços extras)
    df_merged = df.copy()
    df_merged["bairro_pad"] = (
        df_merged[coluna_bairro]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    
    # Criar coluna padronizada na tabela de bairros
    bairros_merge = bairros.copy()
    bairros_merge["bairro_aux"] = (
        bairros_merge["bairro_consolidado"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    
    # Renomear zona para zona_bairro
    if "zona" in bairros_merge.columns:
        bairros_merge = bairros_merge.rename(columns={"zona": "zona_bairro"})
    
    # Fazer merge
    bairros_merge = bairros_merge[["bairro_aux", "zona_bairro"]].drop_duplicates()
    
    df_merged = df_merged.merge(
        bairros_merge,
        left_on="bairro_pad",
        right_on="bairro_aux",
        how="left"
    )
    
    # Remover colunas auxiliares
    if "bairro_pad" in df_merged.columns:
        df_merged = df_merged.drop(columns=["bairro_pad"])
    if "bairro_aux" in df_merged.columns:
        df_merged = df_merged.drop(columns=["bairro_aux"])
    
    return df_merged


def consolidar_zona(df):
    """
    Consolida zona: prioridade para zona do bairro, depois zona da etapa.
    
    Alinhado com o processo manual em nb_gold_pdr_acompannhamento_os.ipynb
    
    Args:
        df: DataFrame com colunas zona_bairro e zona_etapa
    
    Returns:
        DataFrame com coluna zona consolidada
    """
    df_consolidado = df.copy()
    
    if "zona_bairro" in df_consolidado.columns and "zona_etapa" in df_consolidado.columns:
        df_consolidado["zona"] = df_consolidado["zona_bairro"].fillna(df_consolidado["zona_etapa"])
        df_consolidado = df_consolidado.drop(columns=["zona_bairro", "zona_etapa"])
    elif "zona_bairro" in df_consolidado.columns:
        df_consolidado["zona"] = df_consolidado["zona_bairro"]
        df_consolidado = df_consolidado.drop(columns=["zona_bairro"])
    elif "zona_etapa" in df_consolidado.columns:
        df_consolidado["zona"] = df_consolidado["zona_etapa"]
        df_consolidado = df_consolidado.drop(columns=["zona_etapa"])
    
    return df_consolidado


def aplicar_merge_prazo_bairros(acto):
    """Mescla dados de prazo e bairros com a base de solicitações."""
    prazo = processar_prazo()
    bairros = processar_bairros()
    
    # Mescla dados (JOIN)
    acto_prazo = pd.merge(acto, prazo, on="servico", how="inner")
    acto_prazo = pd.merge(acto_prazo, bairros, on="bairro_consolidado", how="left")
    return acto_prazo



def tratar_datas_prazos(acto_prazo):
    """
    Trata datas e calcula prazos de execução.
    Adaptado do nb_ingest_acto_santos.ipynb
    """
    if acto_prazo.empty:
        return acto_prazo
    
    # Trata datas
    acto_prazo["data_de_solicitacao"] = pd.to_datetime(
        acto_prazo["data_de_solicitacao"], dayfirst=True, errors='coerce'
    )
    acto_prazo["data_de_finalizacao"] = pd.to_datetime(
        acto_prazo["data_de_finalizacao"], dayfirst=True, errors='coerce'
    )

    # Calcula vencimento do prazo
    if "prazo_de_conclusao" in acto_prazo.columns:
        acto_prazo["data_de_vencimento_prazo"] = acto_prazo[
            "data_de_solicitacao"
        ] + pd.to_timedelta(acto_prazo["prazo_de_conclusao"], unit="D")

    # Calcula tempo de execução se finalizado
    if "status" in acto_prazo.columns:
        acto_prazo["tempo_de_execucao_real"] = np.where(
            acto_prazo["status"] == "Finalizado",
            (acto_prazo["data_de_finalizacao"] - acto_prazo["data_de_solicitacao"]).dt.days,
            0,
        )
        acto_prazo["tempo_de_execucao_real"] = (
            acto_prazo["tempo_de_execucao_real"].astype(int)
        )

    # Data de hoje como datetime apenas com data (sem hora)
    data_hoje = pd.to_datetime(datetime.now().date())

    # Calcula dias até vencimento
    if "data_de_vencimento_prazo" in acto_prazo.columns:
        acto_prazo["dias_ate_vencimento"] = (
            acto_prazo["data_de_vencimento_prazo"] - data_hoje
        ).dt.days

        if "status" in acto_prazo.columns:
            acto_prazo["dias_ate_vencimento"] = np.where(
                acto_prazo["status"].isin(["Finalizado", "Cancelado"]),
                0,
                acto_prazo["dias_ate_vencimento"]
            )

    # Campos de mês e ano
    if "data_de_solicitacao" in acto_prazo.columns:
        acto_prazo['mes_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.month
        acto_prazo['ano_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.year

    if "data_de_finalizacao" in acto_prazo.columns:
        acto_prazo['mes_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.month
        acto_prazo['ano_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.year

    if "data_de_vencimento_prazo" in acto_prazo.columns:
        acto_prazo['mes_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.month
        acto_prazo['ano_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.year

    # Define status do prazo de execução
    if "data_de_vencimento_prazo" in acto_prazo.columns and "status" in acto_prazo.columns:
        data_hoje_normalizada = pd.to_datetime(datetime.now().date())
        data_vencimento_normalizada = pd.to_datetime(acto_prazo["data_de_vencimento_prazo"]).dt.date
        data_finalizacao_normalizada = pd.to_datetime(acto_prazo["data_de_finalizacao"]).dt.date

        acto_prazo["status_conclusao_servico"] = np.where(
            acto_prazo["status"].str.lower() == "cancelado",
            "Cancelado",
            np.where(
                acto_prazo["data_de_finalizacao"].notna(),
                np.where(
                    data_finalizacao_normalizada <= data_vencimento_normalizada,
                    "Dentro do prazo",
                    "Fora do prazo"
                ),
                np.where(
                    data_vencimento_normalizada > data_hoje_normalizada.date(),
                    "Dentro do prazo",
                    np.where(
                        data_vencimento_normalizada == data_hoje_normalizada.date(),
                        "Vence hoje",
                        "Vencido"
                    )
                )
            )
        )

    # Converte datas para date (sem hora)
    for date_col in ['data_de_solicitacao', 'data_de_finalizacao', 'data_de_vencimento_prazo']:
        if date_col in acto_prazo.columns:
            acto_prazo[date_col] = pd.to_datetime(acto_prazo[date_col]).dt.date

    return acto_prazo


def tratar_base_final_solicitacoes(acto_prazo):
    """
    Tratamentos finais na base de solicitações.
    Adaptado do nb_ingest_acto_santos.ipynb
    """
    # Atribui unidade executora (COPAISA ou região)
    servicos_copaisa = [
        "Corte de grama",
        "Avaliação técnica de árvores",
        "Poda de copa de árvore",
        "Poda de raiz de árvore",
        "Remoção de árvores",
    ]

    if "nome_do_servico_avaliado" in acto_prazo.columns:
        acto_prazo["unidade_executora"] = np.where(
            (acto_prazo["nome_do_servico_avaliado"].isin(servicos_copaisa))
            | (acto_prazo["servico"].isin(servicos_copaisa)) if "servico" in acto_prazo.columns else False,
            "COPAISA",
            np.where(
                acto_prazo["secretaria"].str.contains("CET") if "secretaria" in acto_prazo.columns else False,
                acto_prazo["secretaria"] if "secretaria" in acto_prazo.columns else "",
                acto_prazo["regiao"] if "regiao" in acto_prazo.columns else "",
            ),
        )
    else:
        if "servico" in acto_prazo.columns:
            acto_prazo["unidade_executora"] = np.where(
                (acto_prazo["servico"].isin(servicos_copaisa)),
                "COPAISA",
                np.where(
                    acto_prazo["secretaria"].str.contains("CET") if "secretaria" in acto_prazo.columns else False,
                    acto_prazo["secretaria"] if "secretaria" in acto_prazo.columns else "",
                    acto_prazo["regiao"] if "regiao" in acto_prazo.columns else "",
                ),
            )

    if "secretaria" in acto_prazo.columns:
        acto_prazo["unidade_executora"] = np.where(
            acto_prazo["secretaria"].str.contains('SEGOV'),
            "SEALURB",
            acto_prazo["unidade_executora"]
        )

    # Caso ainda reste algum valor em branco, atribui rótulo padrão
    if "unidade_executora" in acto_prazo.columns:
        acto_prazo["unidade_executora"] = acto_prazo["unidade_executora"].fillna(
            "Não informado"
        )

    # Define responsavel pela execução dos serviços da SEINFRA
    if "responsavel_execucao" in acto_prazo.columns and "etapa_atual" in acto_prazo.columns:
        acto_prazo["responsavel_execucao"] = np.where(
            acto_prazo['etapa_atual'] == 'Execução Terceiro',
            "Empresa terceira",
            acto_prazo["responsavel_execucao"]
        )
        
        if "secretaria" in acto_prazo.columns and "status" in acto_prazo.columns:
            acto_prazo["responsavel_execucao"] = np.where(
                acto_prazo["secretaria"].str.contains("SEINFRA")
                & acto_prazo["status"].isin(["Em atendimento", "Pendente"])
                & acto_prazo["responsavel_execucao"].isna(),
                "A ser definido",
                acto_prazo["responsavel_execucao"]
            )
            acto_prazo["responsavel_execucao"] = np.where(
                acto_prazo["secretaria"].str.contains("SEINFRA")
                & ~acto_prazo["status"].isin(["Em atendimento", "Pendente"])
                & acto_prazo["responsavel_execucao"].isna(),
                "Execução própria",
                acto_prazo["responsavel_execucao"]
            )

    return acto_prazo


def remover_registros_teste(acto_prazo):
    """
    Remove registros de solicitantes testes ou indevidos.
    Adaptado do nb_ingest_acto_santos.ipynb
    """
    if "solicitante" not in acto_prazo.columns:
        return acto_prazo
    
    lista = [
        "André Ygor Bulata Dos Santos",
        "Matheus De Paula Moura Arsenes",
        "Wagner De Morais Pechim",
        "Teste M",
        "Teste Matheus",
        "Teste Teste",
        "Dialla Araujo Souza",
        "Victor Martins Da Silva",
        "Yago Silva De Jesus",
        "Matheus Santos Alves",
        "Milena Firmiano Lopes",
        "Arleque Sandra Aparecida De Souza",
    ]

    acto_prazo["solicitante"] = acto_prazo["solicitante"].str.strip().str.title()
    acto_prazo = acto_prazo.loc[~acto_prazo["solicitante"].isin(lista)]

    return acto_prazo

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 10, Finished, Available, Finished)

In [9]:
# ==========================================
# VALIDAÇÃO DO PAYLOAD (OPCIONAL)
# ==========================================
# Execute esta célula para validar o payload_json_completo

info_payload = validar_payload(payload_json_completo)

if info_payload.get("valido"):
    print("✅ Payload válido!")
    print(f"   Nome: {info_payload.get('nome')}")
    print(f"   ID: {info_payload.get('id')}")
    print(f"   Ativo: {info_payload.get('ativo')}")
    print(f"   Total de solicitações: {info_payload.get('total_solicitacoes')}")
    print(f"   Total de códigos de catálogo: {info_payload.get('total_cod_catalogo')}")
    print(f"\n📋 Códigos de catálogo encontrados:")
    for i, cod in enumerate(info_payload.get('cod_catalogos', []), 1):
        print(f"   {i}. {cod}")
else:
    print(f"❌ Payload inválido: {info_payload.get('erro')}")

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 11, Finished, Available, Finished)

✅ Payload válido!
   Nome: Base_obras_santos
   ID: 696a5bbc1c7f6221017ae73f
   Ativo: True
   Total de solicitações: 1
   Total de códigos de catálogo: 26

📋 Códigos de catálogo encontrados:
   1. 4803
   2. 4804
   3. 5605
   4. 5625
   5. 5626
   6. 5627
   7. 5628
   8. 5677
   9. 5679
   10. 5685
   11. 5686
   12. 5693
   13. 5725
   14. 5755
   15. 5964
   16. 6093
   17. 6113
   18. 6326
   19. 6383
   20. 6513
   21. 6738
   22. 6783
   23. 6963
   24. 7523
   25. 8134
   26. 12804


In [10]:
# ==========================================
# EXEMPLO DE USO - CRIAR TABELA DE SOLICITAÇÕES
# ==========================================
# Use a função criar_tabela_solicitacoes_completa() para criar a tabela completa

# Exemplo básico (sem logs):
# df_solicitacoes = criar_tabela_solicitacoes_completa()

# Exemplo com logs detalhados:
# df_solicitacoes = criar_tabela_solicitacoes_completa(verbose=True)

# Exemplo com códigos específicos:
# cod_catalogo_custom = [4803, 4804, 5605]
# df_solicitacoes = criar_tabela_solicitacoes_completa(cod_catalogo=cod_catalogo_custom, verbose=True)

# ==========================================
# EXEMPLO DE USO - FUNÇÕES INDIVIDUAIS
# ==========================================
# Exemplos de uso das funções principais:

# 1. Extrair códigos de catálogo:
# cod_catalogo = extrair_cod_catalogo(payload_json_completo, verbose=True)

# 2. Buscar dados de etapas:
# df_etapas = obter_dados_etapa_atual(TOKEN, cod_catalogo, verbose=True)

# 3. Buscar dados de solicitações:
# df_solicitacoes = fetch_tabela(payload_json_completo, verbose=True)

# 4. Adicionar etapa atual:
# df_solicitacoes_completo = adicionar_etapa_atual_2(df_etapas, df_solicitacoes)

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 12, Finished, Available, Finished)

In [11]:
# ==========================================
# EXTRAIR CÓDIGOS DE CATÁLOGO (OPCIONAL)
# ==========================================
# Execute esta célula para extrair e visualizar códigos do payload

cod_catalogo_lista = extrair_cod_catalogo(payload_json_completo, verbose=True)

if cod_catalogo_lista:
    print(f"\n📋 Lista completa de codCatalogo:")
    for i, cod in enumerate(sorted(cod_catalogo_lista), 1):
        print(f"   {i}. {cod}")
    
    # Armazena na variável para uso posterior
    cod_catalogo = cod_catalogo_lista
    print(f"\n💾 Códigos armazenados na variável 'cod_catalogo'")
else:
    print("\n❌ Nenhum código foi extraído. Verifique o payload_json_completo.")

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 13, Finished, Available, Finished)

✅ Encontrados 26 códigos de catálogo

📋 Lista completa de codCatalogo:
   1. 4803
   2. 4804
   3. 5605
   4. 5625
   5. 5626
   6. 5627
   7. 5628
   8. 5677
   9. 5679
   10. 5685
   11. 5686
   12. 5693
   13. 5725
   14. 5755
   15. 5964
   16. 6093
   17. 6113
   18. 6326
   19. 6383
   20. 6513
   21. 6738
   22. 6783
   23. 6963
   24. 7523
   25. 8134
   26. 12804

💾 Códigos armazenados na variável 'cod_catalogo'


In [12]:
# ==========================================
# VALIDAÇÃO DA API (OPCIONAL)
# ==========================================
# Execute esta célula para verificar se a API está respondendo corretamente

print("🔍 VALIDAÇÃO DA API")
print("=" * 60)

# 1. Verificar se TOKEN está configurado
token = TOKEN_SANTOS_OBRAS if 'TOKEN_SANTOS_OBRAS' in globals() else None
if token and len(token) > 20:
    print("✅ TOKEN: Configurado")
else:
    print("❌ TOKEN: Não configurado ou inválido")
    print("   Execute a célula 0 para carregar o token do config_api_acto.ipynb")

# 2. Testar listagem de catálogos
print("\n📋 Testando listagem de catálogos...")
try:
    df_catalogos = listar_catalogos(verbose=True)
    if not df_catalogos.empty and 'codCatalogo' in df_catalogos.columns:
        print(f"   Primeiros 5 códigos: {list(df_catalogos['codCatalogo'].head(5).values)}")
except Exception as e:
    print(f"   ❌ Erro: {str(e)[:200]}")

# 3. Testar obter dados de etapas
if token:
    print("\n📥 Testando obtenção de dados de etapas...")
    try:
        codigo_teste = [4803]
        df_etapas = obter_dados_etapa_atual(token, codigo_teste, verbose=True)
        if not df_etapas.empty:
            print(f"   Colunas: {list(df_etapas.columns)[:5]}...")
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:200]}")

print("=" * 60)

StatementMeta(, c2e1059b-cf04-4d87-a394-a67210233f8e, 14, Finished, Available, Finished)

🔍 VALIDAÇÃO DA API
✅ TOKEN: Configurado

📋 Testando listagem de catálogos...
✅ Encontrados 49 catálogos disponíveis
   Primeiros 5 códigos: [6383, 6963, 13004, 6723, 5755]

📥 Testando obtenção de dados de etapas...
✅ DataFrame criado: 10978 linhas, 15 colunas
   Colunas: ['codEtapa', 'etapa', 'servico', 'seqFluxo', 'dataCriacaoOS']...
